# НСПД: максимальный сбор данных по объектам

Ноутбук получает публичные сведения НСПД по зданию, указанному в полном адресе объекта из Сферы.
Полный адрес сначала разбирается через CDI. Квартира, офис, комната или помещение в запрос НСПД не передаются.

На выходе создаётся один датасет:

- одна строка соответствует одной исходной строке Сферы
- если найдено одно здание, его сведения присоединяются
- если найдено несколько зданий, сведения не присоединяются и ставится `ambiguous`
- если здание не найдено, поля НСПД остаются пустыми

Сначала запускается пилот на небольшом количестве строк. Полный запуск включается отдельно.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import random
import re
import time

import pandas as pd
import requests
try:
    import oracledb
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        'Не установлен oracledb. Выполни в отдельной ячейке: %pip install oracledb'
    ) from error
from IPython.display import display

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 120)

## 1. Настройки

Входной файл положи в папку `результаты/НСПД` и назови `nspd_input.csv`.

В нём должна быть колонка с полным адресом:

- `full_address`, `sphere_full_address` или `Полный адрес`

Дополнительно можно оставить `object_id`, `contract_id`, площадь, страховую сумму и другие исходные поля. Они сохранятся в результате.

In [ ]:
def find_project_root():
    current = Path.cwd().resolve()
    for folder in [current, *current.parents]:
        if (folder / 'AGENTS.md').exists():
            return folder
    raise FileNotFoundError('Не найдена корневая папка проекта с AGENTS.md')


PROJECT_ROOT = find_project_root()
OUTPUT_DIR = PROJECT_ROOT / 'результаты' / 'НСПД'
INPUT_FILE = OUTPUT_DIR / 'nspd_input.csv'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# сначала оставь False: будет обработано только PILOT_LIMIT уникальных адресов
FULL_RUN = False
PILOT_LIMIT = 20

# дополнительные вкладки дают больше данных, но увеличивают число запросов
COLLECT_EXTRA_TABS = False

# пауза нужна, чтобы не создавать лишнюю нагрузку на сайт
REQUEST_DELAY_SECONDS = 2.5
REQUEST_JITTER_SECONDS = 1.0
MAX_RETRIES = 4
REQUEST_TIMEOUT_SECONDS = 45

# оставь True; False используй только при ошибке сертификата в рабочем контуре
VERIFY_SSL = True

SEARCH_URL = 'https://nspd.gov.ru/api/geoportal/v2/search/geoportal'
# 36049 — здания; помещения, участки и другие слои не запрашиваются
SEARCH_LAYER_IDS = [36049]
OUTPUT_CRS = 'EPSG:4326'
TAB_VALUES_URL = 'https://nspd.gov.ru/api/geoportal/v1/tab-values-data'
TAB_GROUP_URL = 'https://nspd.gov.ru/api/geoportal/v1/tab-group-data'

RAW_FILE = OUTPUT_DIR / 'nspd_raw.jsonl'
TAB_RAW_FILE = OUTPUT_DIR / 'nspd_tabs_raw.jsonl'

print('Входной файл:', INPUT_FILE)
print('Результаты:', OUTPUT_DIR)
print('Режим:', 'полный запуск' if FULL_RUN else f'пилот на {PILOT_LIMIT} уникальных адресах')

## 2. Загрузка входного файла

Ноутбук сам определяет разделитель и кодировку CSV. Повторяющиеся адреса запрашиваются один раз, а затем результат возвращается ко всем исходным строкам.

In [ ]:
def read_csv_flexible(path):
    encodings = ['utf-8-sig', 'utf-8', 'cp1251']
    last_error = None
    for encoding in encodings:
        try:
            return pd.read_csv(path, sep=None, engine='python', encoding=encoding, dtype='string')
        except Exception as error:
            last_error = error
    raise RuntimeError(f'Не удалось прочитать {path}: {last_error}')


if not INPUT_FILE.exists():
    example = pd.DataFrame({
        'object_id': ['example_1'],
        'full_address': ['пример адреса'],
    })
    example.to_csv(OUTPUT_DIR / 'nspd_input_example.csv', sep=';', index=False, encoding='utf-8-sig')
    raise FileNotFoundError(
        f'Не найден {INPUT_FILE}. Создан пример nspd_input_example.csv. '
        'Скопируй его как nspd_input.csv и замени пример своими данными.'
    )

input_df = read_csv_flexible(INPUT_FILE)
input_df.columns = [str(column).strip() for column in input_df.columns]
input_df.insert(0, 'nspd_input_row_id', range(1, len(input_df) + 1))

print('Строк во входном файле:', len(input_df))
display(input_df.head(3))

## 3. Подготовка поискового запроса

Для каждой строки выполняются два шага:

1. исходный `full_address` без изменений передаётся в CDI
2. из ответа CDI собирается адрес здания без квартиры, офиса или помещения
3. в НСПД запрашивается только слой зданий

Дом, корпус и строение сохраняются. Исходный адрес остаётся в результате для проверки. Если CDI не нашёл один ФИАС дома, запрос в НСПД не выполняется.

In [ ]:
def normalized_name(value):
    return re.sub(r'[^a-zа-яё0-9]+', '_', str(value).strip().lower()).strip('_')


def clean_value(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    if not text or text.lower() in {'nan', 'none', 'null'}:
        return None
    return text


column_by_normalized_name = {
    normalized_name(column): column for column in input_df.columns
}
address_aliases = [
    'full_address', 'sphere_full_address', 'source_address',
    'полный_адрес', 'нормализованный_адрес', 'адрес',
]
address_column = next(
    (column_by_normalized_name[name] for name in address_aliases
     if name in column_by_normalized_name),
    None,
)
if address_column is None:
    raise ValueError('Во входном файле нет колонки с полным адресом')

input_df['sphere_full_address_for_cdi'] = input_df[address_column].map(clean_value)
unique_addresses = (
    input_df[['sphere_full_address_for_cdi']]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)
if not FULL_RUN:
    unique_addresses = unique_addresses.head(PILOT_LIMIT).copy()
unique_addresses.insert(0, 'cdi_lookup_id', range(1, len(unique_addresses) + 1))

# учётные данные используются только для вызова CDI
credential_paths = [
    PROJECT_ROOT / 'парсинги' / 'уч данные.txt',
    PROJECT_ROOT / 'notebooks' / 'уч данные.txt',
]
CREDENTIALS_PATH = next((path for path in credential_paths if path.exists()), None)
if CREDENTIALS_PATH is None:
    raise FileNotFoundError('Не найден файл уч данные.txt')

credentials = {}
for line_number, raw_line in enumerate(
    CREDENTIALS_PATH.read_text(encoding='utf-8-sig').splitlines(), start=1
):
    line = raw_line.strip()
    if not line or line.startswith('#'):
        continue
    if '=' not in line:
        raise ValueError(f'Строка {line_number} в уч данные.txt записана без знака =')
    key, value = line.split('=', 1)
    credentials[key.strip()] = value.strip()

required_khd = ['KHD_HOST', 'KHD_SERVICE_NAME', 'KHD_USER', 'KHD_PASSWORD']
missing_khd = [key for key in required_khd if not credentials.get(key)]
if missing_khd:
    raise ValueError('Заполни в уч данные.txt: ' + ', '.join(missing_khd))

khd_dsn = oracledb.makedsn(
    credentials['KHD_HOST'],
    int(credentials.get('KHD_PORT', '1521')),
    service_name=credentials['KHD_SERVICE_NAME'],
)
khd_connection = oracledb.connect(
    user=credentials['KHD_USER'],
    password=credentials['KHD_PASSWORD'],
    dsn=khd_dsn,
)

CDI_SCHEMA = credentials.get('CDI_SCHEMA', 'DM_MOTOR').upper()
CDI_FUNCTION = credentials.get('CDI_TEXT_FUNCTION', 'F_GET_CDI_ADDR_BY_TEXT').upper()
for value in [CDI_SCHEMA, CDI_FUNCTION]:
    if not value.replace('_', '').isalnum():
        raise ValueError('Некорректное имя функции CDI')

cdi_sql_variants = [
    f'select d.* from {CDI_SCHEMA}.{CDI_FUNCTION}(:address_text) d',
    f'select d.* from table({CDI_SCHEMA}.{CDI_FUNCTION}(:address_text)) d',
]
cdi_sql = None
if not unique_addresses.empty:
    test_address = unique_addresses.iloc[0]['sphere_full_address_for_cdi']
    errors = []
    with khd_connection.cursor() as cursor:
        for sql_variant in cdi_sql_variants:
            try:
                cursor.execute(sql_variant, address_text=test_address)
                cursor.fetchmany(1)
                cdi_sql = sql_variant
                break
            except oracledb.Error as error:
                errors.append(str(error))
    if cdi_sql is None:
        raise RuntimeError('Не удалось вызвать DM_MOTOR.F_GET_CDI_ADDR_BY_TEXT:\n' + '\n'.join(errors))

cdi_records = []
cdi_error_records = []
cdi_columns = []
if cdi_sql is not None:
    with khd_connection.cursor() as cursor:
        for number, item in enumerate(unique_addresses.itertuples(index=False), start=1):
            try:
                cursor.execute(cdi_sql, address_text=item.sphere_full_address_for_cdi)
                columns = [str(column[0]).lower() for column in cursor.description]
                cdi_columns = columns
                for values in cursor.fetchall():
                    record = dict(zip(columns, values))
                    record['cdi_lookup_id'] = int(item.cdi_lookup_id)
                    record['sphere_full_address_for_cdi'] = item.sphere_full_address_for_cdi
                    cdi_records.append(record)
            except oracledb.Error as error:
                cdi_error_records.append({
                    'cdi_lookup_id': int(item.cdi_lookup_id),
                    'sphere_full_address_for_cdi': item.sphere_full_address_for_cdi,
                    'cdi_error': str(error),
                })
            if number % 100 == 0:
                print('CDI:', number, 'из', len(unique_addresses))
khd_connection.close()

cdi_raw_df = pd.DataFrame(cdi_records)
cdi_errors_df = pd.DataFrame(cdi_error_records)
house_fias_column = next(
    (name for name in ['house_fias_id', 'fias_id_house', 'fias_house_id']
     if name in cdi_columns),
    None,
)
if house_fias_column is None and not unique_addresses.empty:
    raise ValueError('CDI не вернул колонку ФИАС дома. Получены: ' + ', '.join(cdi_columns))

# названия полей CDI могут немного отличаться между версиями функции
component_aliases = {
    'postal_code': ['postal_code', 'index', 'zip_code'],
    'region': ['region_with_type', 'region', 'region_name'],
    'area': ['area_with_type', 'area', 'district'],
    'city': ['city_with_type', 'city'],
    'settlement': ['settlement_with_type', 'settlement', 'locality'],
    'street': ['street_with_type', 'street'],
    'house': ['house', 'house_number', 'house_num'],
    'corpus': ['block', 'korpus', 'corpus'],
    'structure': ['building', 'stroenie', 'structure'],
}


def value_by_alias(record, aliases):
    for alias in aliases:
        value = clean_value(record.get(alias))
        if value is not None:
            return value
    return None


def remove_premise_from_address(address):
    # удаляется только квартира, офис, комната или помещение и всё после них
    pattern = (
        r'(?i)(?:[,;]\s*|\s+)'
        r'(?:кв(?:артира)?|пом(?:ещение)?|офис|комн(?:ата)?|апартамент(?:ы)?)'
        r'(?:\.|\s|№|#).*?$'
    )
    result = re.sub(pattern, '', address).strip(' ,;')
    return re.sub(r'\s+', ' ', result) or None


def build_address_of_building(record, source_address):
    values = {
        name: value_by_alias(record, aliases)
        for name, aliases in component_aliases.items()
    }
    parts = []
    for name in ['postal_code', 'region', 'area', 'city', 'settlement', 'street']:
        value = values[name]
        if value and normalized_name(value) not in {normalized_name(x) for x in parts}:
            parts.append(value)
    if values['house']:
        parts.append('д ' + values['house'])
    if values['corpus']:
        parts.append('к ' + values['corpus'])
    if values['structure']:
        parts.append('стр ' + values['structure'])
    if values['house'] and parts:
        return ', '.join(parts), 'cdi_address_parts'
    # CDI подтвердил дом, но не вернул адресные части в ожидаемых колонках
    return remove_premise_from_address(source_address), 'cdi_house_fias_and_trimmed_address'

if cdi_raw_df.empty:
    cdi_raw_df = pd.DataFrame(columns=[
        'cdi_lookup_id', 'sphere_full_address_for_cdi', house_fias_column or 'house_fias_id'
    ])
cdi_raw_df['cdi_house_fias_id'] = (
    cdi_raw_df[house_fias_column].astype('string').str.strip().replace('', pd.NA)
    if house_fias_column else pd.Series(dtype='string')
)

lookup_rows = []
error_ids = set(cdi_errors_df.get('cdi_lookup_id', pd.Series(dtype='int64')))
for item in unique_addresses.itertuples(index=False):
    rows = cdi_raw_df.loc[cdi_raw_df['cdi_lookup_id'].eq(item.cdi_lookup_id)].copy()
    fias_values = rows['cdi_house_fias_id'].dropna().drop_duplicates()
    result = {
        'cdi_lookup_id': int(item.cdi_lookup_id),
        'sphere_full_address_for_cdi': item.sphere_full_address_for_cdi,
        'cdi_candidate_count': len(rows),
        'cdi_house_fias_count': len(fias_values),
        'cdi_house_fias_id': None,
        'cdi_building_address': None,
        'nspd_query_source': None,
        'cdi_status': 'not_found',
    }
    if item.cdi_lookup_id in error_ids:
        result['cdi_status'] = 'lookup_error'
    elif len(fias_values) > 1:
        result['cdi_status'] = 'ambiguous_house_fias'
    elif len(fias_values) == 1:
        fias_id = fias_values.iloc[0]
        chosen = rows.loc[rows['cdi_house_fias_id'].eq(fias_id)].iloc[0].to_dict()
        building_address, query_source = build_address_of_building(
            chosen, item.sphere_full_address_for_cdi
        )
        result.update({
            'cdi_house_fias_id': fias_id,
            'cdi_building_address': building_address,
            'nspd_query_source': query_source,
            'cdi_status': 'unique_house_fias' if building_address else 'no_building_address',
        })
    lookup_rows.append(result)

cdi_lookup_columns = [
    'cdi_lookup_id', 'sphere_full_address_for_cdi', 'cdi_candidate_count',
    'cdi_house_fias_count', 'cdi_house_fias_id', 'cdi_building_address',
    'nspd_query_source', 'cdi_status',
]
cdi_lookup_df = pd.DataFrame(lookup_rows, columns=cdi_lookup_columns)
input_df = input_df.merge(
    cdi_lookup_df.drop(columns=['cdi_lookup_id'], errors='ignore'),
    on='sphere_full_address_for_cdi',
    how='left',
    validate='many_to_one',
)
input_df['cdi_status'] = input_df['cdi_status'].fillna(
    input_df['sphere_full_address_for_cdi'].map(
        lambda value: 'no_source_address' if pd.isna(value) else 'not_run_in_pilot'
    )
)
input_df['nspd_query'] = input_df['cdi_building_address']
input_df['nspd_query_type'] = input_df['nspd_query'].map(
    lambda value: 'cdi_building_address' if clean_value(value) else 'no_query'
)
query_df = (
    input_df.loc[input_df['nspd_query'].notna(), ['nspd_query', 'nspd_query_type']]
    .drop_duplicates()
    .reset_index(drop=True)
)

print('Колонка адреса:', address_column)
print('Адресов передано в CDI:', len(unique_addresses))
print('Уникальных запросов зданий в НСПД:', len(query_df))
display(input_df['cdi_status'].value_counts(dropna=False).rename('строк'))
display(input_df[[
    'sphere_full_address_for_cdi', 'cdi_house_fias_id',
    'cdi_building_address', 'nspd_query_source', 'cdi_status',
]].head(20))

## 4. Функции запроса и сохранения

После каждого ответа результат сразу дописывается в JSONL. Если выполнение остановится, уже полученные ответы не потеряются. При повторном запуске они будут взяты из файла, а повторный запрос на сайт не отправится.

При ответе `429` ноутбук увеличивает паузу. При ответе `403` выполнение останавливается: это защита от продолжения после ограничения доступа.

In [ ]:
session = requests.Session()
session.headers.update({
    'Accept': 'application/json',
    'Accept-Language': 'ru-RU,ru;q=0.9',
    'Referer': 'https://nspd.gov.ru/map?thematic=PKK',
    'User-Agent': 'Mozilla/5.0 NSPD research notebook; low request rate',
})


def query_key(query, query_type):
    layers = ','.join(map(str, SEARCH_LAYER_IDS))
    text = f'{OUTPUT_CRS}|{layers}|{query_type}|{query}'
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


def append_jsonl(path, record):
    with path.open('a', encoding='utf-8') as file:
        file.write(json.dumps(record, ensure_ascii=False, default=str) + '\n')


def load_jsonl(path):
    records = []
    if not path.exists():
        return records
    with path.open('r', encoding='utf-8') as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                print(f'Пропущена повреждённая строка {line_number} в {path.name}')
    return records


class AccessBlockedError(RuntimeError):
    pass


def request_json(url, params):
    last_error = None
    for attempt in range(MAX_RETRIES):
        try:
            response = session.get(
                url,
                params=params,
                timeout=REQUEST_TIMEOUT_SECONDS,
                verify=VERIFY_SSL,
            )

            if response.status_code == 403:
                raise AccessBlockedError(
                    'НСПД вернул 403. Выполнение остановлено. Не продолжай запросы сразу.'
                )

            if response.status_code == 429:
                retry_after = response.headers.get('Retry-After')
                pause = float(retry_after) if retry_after and retry_after.isdigit() else 30 * (attempt + 1)
                print(f'НСПД просит снизить частоту. Пауза {pause:.0f} секунд')
                time.sleep(pause)
                continue

            if response.status_code == 404:
                try:
                    return response.json(), response.status_code, None
                except Exception:
                    return None, response.status_code, None

            if response.status_code >= 500:
                raise requests.HTTPError(f'Ошибка НСПД {response.status_code}')

            response.raise_for_status()
            return response.json(), response.status_code, None

        except AccessBlockedError:
            raise
        except Exception as error:
            last_error = error
            if attempt < MAX_RETRIES - 1:
                time.sleep(5 * (2 ** attempt))

    return None, None, str(last_error)


def polite_pause():
    time.sleep(REQUEST_DELAY_SECONDS + random.uniform(0, REQUEST_JITTER_SECONDS))

## 5. Основной поиск НСПД

Поиск возвращает все найденные карточки. Ноутбук не выбирает первую строку автоматически: количество кандидатов сохраняется отдельно.

In [ ]:
cached_search_records = {
    record['query_key']: record
    for record in load_jsonl(RAW_FILE)
    if record.get('query_key')
}

print('Ответов найдено в локальном кэше:', len(cached_search_records))

for number, row in enumerate(query_df.itertuples(index=False), start=1):
    query = row.nspd_query
    query_type = row.nspd_query_type
    key = query_key(query, query_type)

    if key in cached_search_records:
        continue

    payload, status_code, error = request_json(
        SEARCH_URL,
        params={'layersId': SEARCH_LAYER_IDS, 'query': query},
    )

    record = {
        'query_key': key,
        'query': query,
        'query_type': query_type,
        'output_crs': OUTPUT_CRS,
        'collected_at_utc': datetime.now(timezone.utc).isoformat(),
        'http_status': status_code,
        'error': error,
        'response': payload,
    }
    append_jsonl(RAW_FILE, record)
    cached_search_records[key] = record

    print(f'{number}/{len(query_df)}: ответ сохранён')
    polite_pause()

print('Основной поиск завершён')

## 6. Разворачивание всех полей

Все поля ответа разворачиваются в колонки. Вложенные списки и геометрия остаются в JSON-виде, поэтому исходная информация не теряется.

In [ ]:
def flatten_dict(value, prefix=''):
    result = {}
    if isinstance(value, dict):
        for key, item in value.items():
            new_prefix = f'{prefix}__{key}' if prefix else str(key)
            result.update(flatten_dict(item, new_prefix))
    elif isinstance(value, list):
        result[prefix] = json.dumps(value, ensure_ascii=False)
    else:
        result[prefix] = value
    return result


def get_features(payload):
    if not isinstance(payload, dict):
        return []
    data = payload.get('data') or {}
    features = data.get('features') or []
    return features if isinstance(features, list) else []


def web_mercator_to_lon_lat(x, y):
    if x is None or y is None:
        return None, None
    lon = float(x) * 180.0 / 20037508.34
    lat = math.degrees(2 * math.atan(math.exp(float(y) / 6378137.0)) - math.pi / 2)
    return lon, lat


def geometry_crs_name(geometry):
    return str(((geometry.get('crs') or {}).get('properties') or {}).get('name') or '')


def geometry_is_wgs84(geometry, points):
    crs_name = geometry_crs_name(geometry).upper()
    if '4326' in crs_name:
        return True
    if '3857' in crs_name:
        return False
    return bool(points) and all(abs(x) <= 180 and abs(y) <= 90 for x, y in points)


def geometry_summary(feature):
    geometry = feature.get('geometry') or {}
    coordinates = geometry.get('coordinates')
    points = []

    def collect(value):
        if isinstance(value, list) and len(value) >= 2 and all(isinstance(x, (int, float)) for x in value[:2]):
            points.append((value[0], value[1]))
        elif isinstance(value, list):
            for item in value:
                collect(item)

    collect(coordinates)
    if not points:
        return {'geometry_type': geometry.get('type')}

    xs = [point[0] for point in points]
    ys = [point[1] for point in points]
    center_x = (min(xs) + max(xs)) / 2
    center_y = (min(ys) + max(ys)) / 2
    source_is_wgs84 = geometry_is_wgs84(geometry, points)
    if source_is_wgs84:
        lon, lat = center_x, center_y
    else:
        lon, lat = web_mercator_to_lon_lat(center_x, center_y)
    return {
        'geometry_type': geometry.get('type'),
        'geometry_source_crs': geometry_crs_name(geometry) or 'определено по диапазону координат',
        'bbox_x_min': min(xs),
        'bbox_y_min': min(ys),
        'bbox_x_max': max(xs),
        'bbox_y_max': max(ys),
        'bbox_center_lon_wgs84': lon,
        'bbox_center_lat_wgs84': lat,
    }


candidate_rows = []
log_rows = []

for query in query_df.itertuples(index=False):
    key = query_key(query.nspd_query, query.nspd_query_type)
    record = cached_search_records.get(key, {})
    features = get_features(record.get('response'))
    if record.get('error'):
        match_status = 'error'
    elif not features:
        match_status = 'not_found'
    elif len(features) == 1:
        match_status = 'unique_building'
    else:
        match_status = 'ambiguous'

    log_rows.append({
        'query_key': key,
        'nspd_query': query.nspd_query,
        'nspd_query_type': query.nspd_query_type,
        'candidate_count': len(features),
        'match_status': match_status,
        'http_status': record.get('http_status'),
        'error': record.get('error'),
        'collected_at_utc': record.get('collected_at_utc'),
    })

    for candidate_number, feature in enumerate(features, start=1):
        row = {
            'query_key': key,
            'nspd_query': query.nspd_query,
            'nspd_query_type': query.nspd_query_type,
            'candidate_number': candidate_number,
            'candidate_count': len(features),
            'match_status': match_status,
        }
        row.update(geometry_summary(feature))
        row.update({f'nspd__{key}': value for key, value in flatten_dict(feature).items()})
        candidate_rows.append(row)

search_log_columns = [
    'query_key', 'nspd_query', 'nspd_query_type', 'candidate_count',
    'match_status', 'http_status', 'error', 'collected_at_utc',
]
candidate_base_columns = [
    'query_key', 'nspd_query', 'nspd_query_type', 'candidate_number',
    'candidate_count', 'match_status',
]
search_log_df = pd.DataFrame(log_rows, columns=search_log_columns)
candidates_df = pd.DataFrame(candidate_rows)
if candidates_df.empty:
    candidates_df = pd.DataFrame(columns=candidate_base_columns)

result_base_df = input_df.merge(
    search_log_df,
    on=['nspd_query', 'nspd_query_type'],
    how='left',
)

# строки без запроса НСПД сохраняются вместе с причиной из CDI
status_from_cdi = {
    'no_source_address': 'no_source_address',
    'not_run_in_pilot': 'not_run_in_pilot',
    'lookup_error': 'cdi_lookup_error',
    'not_found': 'cdi_house_not_found',
    'ambiguous_house_fias': 'cdi_house_ambiguous',
    'no_building_address': 'cdi_building_address_not_created',
}
missing_search = result_base_df['match_status'].isna()
result_base_df.loc[missing_search, 'match_status'] = (
    result_base_df.loc[missing_search, 'cdi_status']
    .map(status_from_cdi)
    .fillna('not_searched')
)
result_base_df['candidate_count'] = (
    pd.to_numeric(result_base_df['candidate_count'], errors='coerce')
    .fillna(0)
    .astype('int64')
)

# в итоговый датасет присоединяется только единственное найденное здание
unique_buildings_df = candidates_df.loc[
    candidates_df['match_status'].eq('unique_building')
].drop(columns=[
    'nspd_query', 'nspd_query_type', 'candidate_count', 'match_status'
], errors='ignore')
unique_buildings_df = unique_buildings_df.drop_duplicates('query_key')

final_dataset_df = result_base_df.merge(
    unique_buildings_df,
    on='query_key',
    how='left',
    validate='many_to_one',
)
final_dataset_df['nspd_data_joined'] = (
    final_dataset_df['match_status'].eq('unique_building').astype('int64')
)

if len(final_dataset_df) != len(input_df):
    raise ValueError('После присоединения НСПД изменилось количество строк')

print('Строк во входном файле:', len(input_df))
print('Строк в итоговом датасете:', len(final_dataset_df))
print('Однозначно присоединено зданий:', final_dataset_df['nspd_data_joined'].sum())
display(result_base_df['match_status'].value_counts(dropna=False).rename('строк'))

## 7. Дополнительные вкладки карточки

НСПД хранит часть сведений во вкладках карточки. Ноутбук пробует получить шесть известных типов вкладок и сохраняет ответы целиком. Для неподходящего типа объекта часть вкладок закономерно будет пустой.

Если `COLLECT_EXTRA_TABS = False`, эта ячейка ничего не запрашивает.

In [ ]:
TAB_REQUESTS = [
    ('land_parts', TAB_VALUES_URL, 'landParts'),
    ('land_links', TAB_VALUES_URL, 'landLinks'),
    ('permission_type', TAB_VALUES_URL, 'permissionType'),
    ('composition_land', TAB_VALUES_URL, 'compositionLand'),
    ('build_parts', TAB_VALUES_URL, 'buildParts'),
    ('objects_list', TAB_GROUP_URL, 'objectsList'),
]

cached_tab_keys = {
    record.get('tab_key')
    for record in load_jsonl(TAB_RAW_FILE)
    if record.get('tab_key')
}

if COLLECT_EXTRA_TABS and not candidates_df.empty:
    # вкладки запрашиваются только для однозначно найденных зданий
    unique_query_keys = set(
        search_log_df.loc[
            search_log_df['match_status'].eq('unique_building'), 'query_key'
        ]
    )
    feature_refs = []
    for query in query_df.itertuples(index=False):
        key = query_key(query.nspd_query, query.nspd_query_type)
        if key not in unique_query_keys:
            continue
        record = cached_search_records.get(key, {})
        for feature in get_features(record.get('response')):
            properties = feature.get('properties') or {}
            category_id = properties.get('category')
            options = properties.get('options') or {}
            geometry_id = feature.get('id')
            no_coords = bool(options.get('no_coords'))
            object_document_id = options.get('objdoc_id')
            registers_id = options.get('registers_id')
            has_reference = (
                no_coords and object_document_id is not None and registers_id is not None
            ) or (
                not no_coords and category_id is not None and geometry_id is not None
            )
            if has_reference:
                feature_refs.append({
                    'query_key': key,
                    'category_id': category_id,
                    'geometry_id': geometry_id,
                    'no_coords': no_coords,
                    'object_document_id': object_document_id,
                    'registers_id': registers_id,
                })

    total_requests = len(feature_refs) * len(TAB_REQUESTS)
    print('Максимум дополнительных запросов:', total_requests)
    completed = 0

    for feature_ref in feature_refs:
        key = feature_ref['query_key']
        category_id = feature_ref['category_id']
        geometry_id = feature_ref['geometry_id']
        for tab_name, url, tab_class in TAB_REQUESTS:
            tab_key = f'{key}|{category_id}|{geometry_id}|{tab_name}'
            if tab_key in cached_tab_keys:
                continue

            if feature_ref['no_coords']:
                tab_params = {
                    'tabClass': tab_class,
                    'objdocId': feature_ref['object_document_id'],
                    'registersId': feature_ref['registers_id'],
                }
            else:
                tab_params = {
                    'tabClass': tab_class,
                    'categoryId': category_id,
                    'geomId': geometry_id,
                }

            payload, status_code, error = request_json(url, params=tab_params)
            append_jsonl(TAB_RAW_FILE, {
                'tab_key': tab_key,
                'query_key': key,
                'category_id': category_id,
                'geometry_id': geometry_id,
                'tab_name': tab_name,
                'tab_class': tab_class,
                'collected_at_utc': datetime.now(timezone.utc).isoformat(),
                'http_status': status_code,
                'error': error,
                'response': payload,
            })
            cached_tab_keys.add(tab_key)
            completed += 1
            if completed % 10 == 0:
                print(f'Получено новых ответов вкладок: {completed}')
            polite_pause()
else:
    print('Дополнительные вкладки отключены или кандидатов нет')

## 8. Подготовка таблицы дополнительных вкладок

In [ ]:
tab_rows = []
for record in load_jsonl(TAB_RAW_FILE):
    row = {
        'query_key': record.get('query_key'),
        'category_id': record.get('category_id'),
        'geometry_id': record.get('geometry_id'),
        'tab_name': record.get('tab_name'),
        'tab_class': record.get('tab_class'),
        'http_status': record.get('http_status'),
        'error': record.get('error'),
        'collected_at_utc': record.get('collected_at_utc'),
    }
    response = record.get('response')
    if response is not None:
        row.update({f'tab__{key}': value for key, value in flatten_dict(response).items()})
    tab_rows.append(row)

tabs_df = pd.DataFrame(tab_rows)
print('Строк с ответами дополнительных вкладок:', len(tabs_df))
display(tabs_df.head(3))

## 9. Сохранение результатов

Создаётся один файл `итоговый_датасет_НСПД.csv`. CSV сохраняется с разделителем `;` и кодировкой `utf-8-sig`, чтобы русский текст корректно открывался в Excel.

Исходные JSONL остаются главным полным архивом ответа НСПД.

In [ ]:
# дополнительные вкладки сохраняются внутри итоговой строки одним JSON-полем
unique_result_query_keys = set(
    final_dataset_df.loc[
        final_dataset_df['match_status'].eq('unique_building'), 'query_key'
    ].dropna()
)
tabs_for_unique_buildings = tabs_df.loc[
    tabs_df.get('query_key', pd.Series(dtype='string')).isin(unique_result_query_keys)
].copy() if not tabs_df.empty else pd.DataFrame()

if not tabs_for_unique_buildings.empty:
    tabs_for_dataset = pd.DataFrame([
        {
            'query_key': query_key_value,
            'nspd_extra_tabs_json': json.dumps(
                group.drop(columns=['query_key']).to_dict('records'),
                ensure_ascii=False,
                default=str,
            ),
        }
        for query_key_value, group in tabs_for_unique_buildings.groupby('query_key', dropna=False)
    ])
    final_dataset_df = final_dataset_df.merge(
        tabs_for_dataset, on='query_key', how='left', validate='many_to_one'
    )
else:
    final_dataset_df['nspd_extra_tabs_json'] = pd.NA

if len(final_dataset_df) != len(input_df):
    raise ValueError('В итоговом датасете изменилось количество строк')

final_path = OUTPUT_DIR / 'итоговый_датасет_НСПД.csv'
final_dataset_df.to_csv(
    final_path, sep=';', index=False, encoding='utf-8-sig'
)

summary_df = (
    final_dataset_df.groupby('match_status', dropna=False)
    .size()
    .rename('количество строк')
    .reset_index()
)
summary_df['доля процентов'] = (
    summary_df['количество строк'] / max(len(final_dataset_df), 1) * 100
).round(2)

print('Сохранён один итоговый датасет:', final_path)
print('Строк:', len(final_dataset_df))
print('Однозначно присоединено:', final_dataset_df['nspd_data_joined'].sum())
display(summary_df)
display(final_dataset_df.head(3))

## Как перейти к полному запуску

1. Проверь пилот в `итоговый_датасет_НСПД.csv`.
2. Убедись, что запросы не возвращают `403` или много ошибок.
3. Если дополнительные вкладки нужны, сначала оцени их заполненность по пилоту. Они сильно увеличивают время работы.
4. Поставь `FULL_RUN = True` и перезапусти ноутбук.
5. Не удаляй JSONL-файлы между запусками: это кэш и точка продолжения.

Статус `unique_building` означает, что найдено одно здание и его сведения присоединены. Статус `ambiguous` означает, что найдено несколько зданий: данные НСПД в эту строку не подставляются.